In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import sys
sys.path.append('/tmp/local_scratch/v_neelesh_bisht/3d-cnn/rsna')

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm
from math import ceil
import matplotlib.patches as patches
from scipy import stats
from sklearn.metrics import precision_recall_curve, average_precision_score


from resnet_cams import ResnetCAMS
from dataset.dataset import TrainAndValidateDataset
from resnet import Resnet
from utils import IOU, AUPRC

Orig_img_size = 1024
img_size = 299

In [2]:
device = torch.device('cuda:7' if torch.cuda.is_available() else 'cpu')
torch.cuda.empty_cache()

In [ ]:
train_and_validate_dataset = TrainAndValidateDataset(test_size=0.01, val_size=0.01)

train_loader = train_and_validate_dataset.train_loader
val_loader = train_and_validate_dataset.val_loader
test_loader = train_and_validate_dataset.test_loader

# train_labels = train_and_validate_dataset.train_labels
val_labels = train_and_validate_dataset.val_labels
test_labels = train_and_validate_dataset.test_labels

# print(train_labels.shape)
print("validation size :",val_labels.shape)
print("test size :",test_labels.shape)   

In [ ]:
target_class_idx = 1  # Target class
ref_class_idx = 0  # Reference class

# model = torch.load('./rsna-dataset/model_inception_v3_17092024.pth')
# torch.save(model.state_dict(), './rsna-dataset/model_inception_v3_17092024_dict.pth')
# resnet_model.to(device)

# Load the model weights
resnet_model = Resnet(device=device)
resnet_model.load_state_dict(torch.load('./rsna-dataset/model_inception_v3_17092024_dict.pth', map_location=device))

# last conv_block : model.Mixed_7c

In [ ]:
final_label_arr = [] # value for all_labels
final_feature_arr = [] # value for all_features

embedding = None

def _embedding_hook_fn(module, input, output):
    global embedding 
    embedding = input[0]  # Storing the input to the fc layer
    # print("inside _embedding_hook_fn: ",input[0].shape) ## torch.Size([128, 512])

embedding_hook = resnet_model.model.fc.register_forward_hook(_embedding_hook_fn)

resnet_model.eval()

correct = 0
total = 0  
for images, labels, _ in tqdm(val_loader):
    images = images.to(device)
    labels = labels.to(device)
    predictions = resnet_model(images)
    # print("predictions", predictions)
    # probabilities = torch.softmax(predictions, dim=1)
    # print("predictions", probabilities)
    _, predicted = torch.max(predictions, 1)
    # print("after",  _, predicted)
    total += labels.size(0)
    correct += (labels == predicted).sum()

    for i in range(images.shape[0]):
        class_id = labels[i].cpu().item()
        final_label_arr.append(class_id)

        feature = embedding[i].detach().cpu().numpy()
        if len(final_feature_arr) == 0:
            final_feature_arr = np.expand_dims(feature, axis=0)
        else:
            final_feature_arr = np.concatenate((final_feature_arr, np.expand_dims(feature, axis=0)), axis = 0)

print(f'Val_Acc: {100*correct/total}')

embedding_hook.remove()

In [ ]:
last_conv_layer_name = 'model.Mixed_7c'

final_image_arr = []
final_ground_truth_label_arr = []
final_bbox_arr = []
final_heatmap_arr = {
    "grad_cam_heatmap": [],
    "diff_grad_cam_heatmap": [],
    "diff_cam_heatmap": [],
    "counter_factual_heatmap": [],
    "torch_cam_grad_cam_heatmap": [],
    "torch_cam_grad_campp_heatmap": [],
    "torch_cam_score_cam_heatmap": []
}
final_cam_wise_probability_arr = {
    "grad_cam_heatmap": [],
    "diff_grad_cam_heatmap": [],
    "diff_cam_heatmap": [],
    "counter_factual_heatmap": [],
    "torch_cam_grad_cam_heatmap": [],
    "torch_cam_grad_campp_heatmap": [],
    "torch_cam_score_cam_heatmap": []
}
heatmap_iou_obj = {
        "grad_cam_heatmap":[], 
        "diff_grad_cam_heatmap": [],
        "diff_cam_heatmap": [], 
        "counter_factual_heatmap": [],
        "torch_cam_grad_cam_heatmap": [],
        "torch_cam_grad_campp_heatmap": [],
        "torch_cam_score_cam_heatmap": []
}
heatmap_auprc_obj = {
        "grad_cam_heatmap":[], 
        "diff_grad_cam_heatmap": [],
        "diff_cam_heatmap": [], 
        "counter_factual_heatmap": [],
        "torch_cam_grad_cam_heatmap": [],
        "torch_cam_grad_campp_heatmap": [],
        "torch_cam_score_cam_heatmap": []
}
np.set_printoptions(precision=4, suppress=True)
methods = ["grad_cam_heatmap", "diff_grad_cam_heatmap", "diff_cam_heatmap", "counter_factual_heatmap", "torch_cam_grad_cam_heatmap", "torch_cam_grad_campp_heatmap", "torch_cam_score_cam_heatmap"]

for images, labels, bboxs in tqdm(test_loader):
    images = images.to(device)
    labels = labels.to(device)
    
    # Iterate over each image in the batch
    for i in range(images.size(0)):
        # Select a single image and label
        image = images[i:i+1]  # Keep batch dimension (1, C, H, W)
        label = labels[i:i+1].cpu().item()

        bbox_coords = []
        for idx, bbox in enumerate(bboxs):
            # Convert tensor to a list or numpy array if needed
            coords_arr = bbox.tolist() 
            bbox_coords.append(coords_arr[i])

        # filter the images with atleast one bounding box for calculating IOU.
        if np.isnan(bbox_coords[0]) or np.isnan(bbox_coords[1]) or np.isnan(bbox_coords[2]) or np.isnan(bbox_coords[3]):
            continue
        
        bbox_coords = [int(x) for x in bbox_coords]

        final_image_arr.append(image)
        final_ground_truth_label_arr.append(label)
        final_bbox_arr.append(bbox_coords)

        cams = ResnetCAMS.get_cams(image, resnet_model, last_conv_layer_name, target_class_idx, ref_class_idx, final_feature_arr=final_feature_arr, final_label_arr=final_label_arr)

        for type in methods:
            heatmap = cams[type]["heatmap"]
            iou = IOU.compute_iou(bbox_coords, cams[type]["heatmap"])
            auprc = AUPRC.compute_auprc(bbox_coords, cams[type]["heatmap"])
            
            output_probs = cams[type]["output"].cpu().detach().numpy()[0]
            output_probs = np.round(output_probs, 4)
            final_heatmap_arr[type].append(heatmap)
            heatmap_iou_obj[type].append(iou)
            heatmap_auprc_obj[type].append(auprc)
            final_cam_wise_probability_arr[type].append(output_probs)


In [ ]:
print(len(final_image_arr))
# print(final_image_arr[0].shape)
# print(final_bbox_arr[0])

# for type in methods:
#     print(f"type {type}: ", final_cam_wise_probability_arr[type])
    # probs = [item[1] for item in final_cam_wise_probability_arr[heatmap_type]]

In [ ]:
numrows = len(final_image_arr)

# Create a single figure with subplots
fig, ax = plt.subplots(numrows, 8, figsize=(30, 5 * (numrows + 1)))

# Loop through each slice and plot the images in the appropriate subplot
for slice_idx in range(0, numrows):

    # Adding titles for the first row
    if slice_idx == 0:
        ax[slice_idx, 0].set_title('Sample')
        ax[slice_idx, 1].set_title('Mask')
        ax[slice_idx, 2].set_title('Grad-CAM')
        ax[slice_idx, 3].set_title('Diff Grad-CAM')
        ax[slice_idx, 4].set_title('Diff-CAM')
        ax[slice_idx, 5].set_title('Counter Factual')
        ax[slice_idx, 6].set_title('Torch-CAM Grad-CAM')
        ax[slice_idx, 7].set_title('Torch-CAM Grad-CAM++')

    final_image = final_image_arr[slice_idx]
    final_image = final_image.cpu().numpy()
    final_image = final_image[0]
    img = np.transpose(final_image, (1, 2, 0))

    box = final_bbox_arr[slice_idx]
    # 'r' means relative. 'c' means center.
    rx = ceil(box[0]*img_size/Orig_img_size) if not np.isnan(box[0]) else 0
    ry = ceil(box[1]*img_size/Orig_img_size) if not np.isnan(box[1]) else 0
    rw = ceil(box[2]*img_size/Orig_img_size) if not np.isnan(box[2]) else 0
    rh = ceil(box[3]*img_size/Orig_img_size) if not np.isnan(box[3]) else 0

    grad_cam_heatmap = final_heatmap_arr["grad_cam_heatmap"][slice_idx]
    diff_grad_cam_heatmap = final_heatmap_arr["diff_grad_cam_heatmap"][slice_idx]
    diff_cam_heatmap = final_heatmap_arr["diff_cam_heatmap"][slice_idx]
    counter_factual_heatmap = final_heatmap_arr["counter_factual_heatmap"][slice_idx]
    torch_cam_grad_cam_heatmap = final_heatmap_arr["torch_cam_grad_cam_heatmap"][slice_idx]
    torch_cam_grad_campp_heatmap = final_heatmap_arr["torch_cam_grad_campp_heatmap"][slice_idx]
    # grad_cam_heatmap = final_heatmap_arr["grad_cam_heatmap"][slice_idx]

    ax[slice_idx, 0].imshow(img, origin='upper', cmap='bone')
    ax[slice_idx, 0].set_xlabel(f'Image {slice_idx + 1}, gt: {final_ground_truth_label_arr[slice_idx]}')

    # TODO: FIT SCORE CAM HERE
    # ax[slice_idx, 1].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))

    img3 = ax[slice_idx, 2].imshow(img, cmap='bone')
    img4 = ax[slice_idx, 2].imshow(grad_cam_heatmap, cmap='jet', alpha=0.5, extent=img3.get_extent())
    img5 = ax[slice_idx, 2].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 2].set_xlabel(f'''Grad-CAM, iou: {heatmap_iou_obj["grad_cam_heatmap"][slice_idx]},
    probs: {final_cam_wise_probability_arr["grad_cam_heatmap"][slice_idx]}, auprc: {heatmap_auprc_obj["grad_cam_heatmap"][slice_idx]} ''')

    img6 = ax[slice_idx, 3].imshow(img, cmap='bone')
    img7 = ax[slice_idx, 3].imshow(diff_grad_cam_heatmap, cmap='jet', alpha=0.5, extent=img6.get_extent())
    img8 = ax[slice_idx, 3].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 3].set_xlabel(f'''Diff Grad-CAM, iou: {heatmap_iou_obj["diff_grad_cam_heatmap"][slice_idx]},
    probs: {final_cam_wise_probability_arr["diff_grad_cam_heatmap"][slice_idx]}, auprc: {heatmap_auprc_obj["grad_cam_heatmap"][slice_idx]} ''')

    img9 = ax[slice_idx, 4].imshow(img, cmap='bone')
    img10 = ax[slice_idx, 4].imshow(diff_cam_heatmap, cmap='jet', alpha=0.5, extent=img9.get_extent())
    img11 = ax[slice_idx, 4].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 4].set_xlabel(f'''Diff-CAM, iou: {heatmap_iou_obj["diff_cam_heatmap"][slice_idx]},
    probs: {final_cam_wise_probability_arr["diff_cam_heatmap"][slice_idx]}, auprc: {heatmap_auprc_obj["grad_cam_heatmap"][slice_idx]} ''')

    img12 = ax[slice_idx, 5].imshow(img, cmap='bone')
    img13 = ax[slice_idx, 5].imshow(counter_factual_heatmap, cmap='jet', alpha=0.5, extent=img12.get_extent())
    img14 = ax[slice_idx, 5].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 5].set_xlabel(f'''Counter Factual, iou: {heatmap_iou_obj["counter_factual_heatmap"][slice_idx]},
    probs: {final_cam_wise_probability_arr["counter_factual_heatmap"][slice_idx]}, auprc: {heatmap_auprc_obj["grad_cam_heatmap"][slice_idx]} ''')

    img15 = ax[slice_idx, 6].imshow(img, cmap='bone')
    img16 = ax[slice_idx, 6].imshow(torch_cam_grad_cam_heatmap, cmap='jet', alpha=0.5, extent=img15.get_extent())
    img17 = ax[slice_idx, 6].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 6].set_xlabel(f'''Torch-CAM Grad-CAM, iou: {heatmap_iou_obj["torch_cam_grad_cam_heatmap"][slice_idx]},
    probs: {final_cam_wise_probability_arr["torch_cam_grad_cam_heatmap"][slice_idx]}, auprc: {heatmap_auprc_obj["grad_cam_heatmap"][slice_idx]} ''')

    img18 = ax[slice_idx, 7].imshow(img, cmap='bone')
    img19 = ax[slice_idx, 7].imshow(torch_cam_grad_campp_heatmap, cmap='jet', alpha=0.5, extent=img18.get_extent())
    img20 = ax[slice_idx, 7].add_patch(patches.Rectangle((rx, ry), rw, rh, linewidth=1, edgecolor='r', facecolor='none'))
    ax[slice_idx, 7].set_xlabel(f'''Torch-CAM Grad-CAM++, iou: {heatmap_iou_obj["torch_cam_grad_campp_heatmap"][slice_idx]},
    probs: {final_cam_wise_probability_arr["torch_cam_grad_campp_heatmap"][slice_idx]}, auprc: {heatmap_auprc_obj["grad_cam_heatmap"][slice_idx]} ''')

# plt.savefig('rsna_output.tiff', bbox_inches='tight', pad_inches=0, dpi=300)
# Adjust the layout
plt.tight_layout()
plt.show()



In [ ]:
def get_correlation_score(arr1=[], arr2=[]):
    score = stats.pearsonr(arr1, arr2)
    return score.statistic

def compute_mean_iou_for_heatmaps(heatmap_iou_obj, heatmap_auprc_obj):
    """Calculate the mean IoU for each type of heatmap."""
    mean_iou = {}

    # Iterate over each type of heatmap
    for tup1, tup2 in list(zip(heatmap_iou_obj.items(), heatmap_auprc_obj.items())):
        heatmap_type, iou_list = tup1
        _, auprc_list = tup2
        # Check if the IoU list is not empty
        probs = [item[1] for item in final_cam_wise_probability_arr[heatmap_type]]
        if iou_list:
            mean_iou[heatmap_type] = {'iou': np.mean(iou_list), 'auprc': np.mean(auprc_list), 'correlation_score': get_correlation_score(iou_list, probs)}
        else:
            mean_iou[heatmap_type] = {'iou': None}  # Use None to indicate no IoU values available
    
    return mean_iou

mean_iou = compute_mean_iou_for_heatmaps(heatmap_iou_obj, heatmap_auprc_obj)


for heatmap_type, obj in mean_iou.items():
    iou = obj["iou"]
    auprc = obj["auprc"]
    correlation_score = obj["correlation_score"]
    print(f"IOU for {heatmap_type}: ", iou*100,'%')
    print(f"AUPRC for {heatmap_type}: ", auprc)
    print(f"Correlation Score for {heatmap_type}: ", correlation_score)
    print('\n')


In [13]:
# Here’s a more detailed color mapping from the jet colormap:

# Blue: Low values (0 to ~0.25) (0 to 63)
# Cyan: Mid-low values (~0.25 to ~0.45) (63 to 114)
# Green: Mid values (~0.45 to ~0.55) > (~114 to ~140)
# Yellow: High intermediate values (~0.55 to ~0.75)
# Orange: Very high intermediate values (~0.75 to ~0.85)
# Red: Maximum values (~0.85 to 1)

#ASSUMPTIONS of threshold here
HEATMAP_THRESHOLD = 114



#### INCEPTIONNET 18th SEPTEMBER FINDINGS:
# Image: inceptionv3_18092024
# 1. IOU increased from 7% (approx) to 11-12% (approx)

# IOU for grad_cam_heatmap:  11.697754858406926 %
# Correlation Score for grad_cam_heatmap:  0.3132867484117334


# IOU for diff_grad_cam_heatmap:  12.065014122398345 %
# Correlation Score for diff_grad_cam_heatmap:  0.3209751156211386


# IOU for diff_cam_heatmap:  10.71177395058959 %
# Correlation Score for diff_cam_heatmap:  0.323337572627577


# IOU for counter_factual_heatmap:  11.265734711474387 %
# Correlation Score for counter_factual_heatmap:  0.36766111451430233


# IOU for torch_cam_grad_cam_heatmap:  11.698569123511023 %
# Correlation Score for torch_cam_grad_cam_heatmap:  0.31320363578912663


# IOU for torch_cam_grad_campp_heatmap:  11.776119604973054 %
# Correlation Score for torch_cam_grad_campp_heatmap:  0.3097849389135056


# IOU for torch_cam_score_cam_heatmap:  11.697754858406926 %
# Correlation Score for torch_cam_score_cam_heatmap:  0.3132867484117334



In [ ]:
# last_conv_layer_name = 'model'
# # last_conv_layer = getattr(getattr(resnet_model, last_conv_layer_name), 'layer4')
# # print(last_conv_layer[2])

# # last_conv_layer = getattr(getattr(getattr(resnet_model, last_conv_layer_name), 'layer4')[2], 'conv3')
# # last_conv_layer = getattr(resnet_model, 'model.layer4.2.conv3')
# last_conv_layer = getattr(getattr(getattr(getattr(resnet_model, 'model'), 'layer4'), '2'), 'conv3')
# print(last_conv_layer)
# print(resnet_model.model.fc)

# for name, layer in resnet_model.named_modules():
#     print(name)

In [ ]:
# import numpy as np
# import torch
# import faiss  
# from utils import CommonUtils

# from torchcam.methods import GradCAM, GradCAMpp, ScoreCAM
# import numpy as np
# import torch
# from resnet_cams import GradCam

# shape = [299,299]
# last_conv_layer_name = 'model.Mixed_7c'

# class TorchCAM():

#     @staticmethod
#     def compute_grad_cam(img_tensor, model, last_conv_layer_name='layer4', target_class_idx=1):
#         model.eval()
#         grad_cam = GradCAM(model, last_conv_layer_name)
#         output = model(img_tensor)
#         output_probabilities = torch.softmax(output, dim=1)
#         heatmaps = []
#         for i in range(img_tensor.size(0)):
#             grad_cams = grad_cam(class_idx=target_class_idx, scores=output[i:i+1])
#             heatmap = grad_cams[0].cpu().numpy().squeeze()
#             heatmaps.append(heatmap)
#         return heatmaps, output_probabilities
    
#     @staticmethod
#     def compute_grad_campp(img_tensor, model, last_conv_layer_name='layer4', target_class_idx=1):
#         model.eval()
#         grad_campp = GradCAMpp(model, last_conv_layer_name)
#         output = model(img_tensor)
#         output_probabilities = torch.softmax(output, dim=1)
#         heatmaps = []
#         for i in range(img_tensor.size(0)):
#             grad_campps = grad_campp(class_idx=target_class_idx, scores=output[i:i+1])
#             heatmap = grad_campps[0].cpu().numpy().squeeze()
#             heatmaps.append(heatmap)
#         return heatmaps, output_probabilities
    
#     @staticmethod
#     def compute_score_cam(img_tensor, model, last_conv_layer_name='layer4', target_class_idx=1):
#         model.eval()
#         score_cam = ScoreCAM(model, last_conv_layer_name)
#         with torch.no_grad(): # TODO: think on this
#             output = model(img_tensor)
#         heatmaps = []
#         for i in range(img_tensor.size(0)):
#             score_cams = score_cam(class_idx=target_class_idx)
#             heatmap = score_cams[0].cpu().numpy().squeeze()
#             heatmaps.append(heatmap)
#         return heatmaps
    
# def get_cams(image, resnet_model, last_conv_layer_name, target_class_idx, ref_class_idx, show_cams=False, final_feature_arr = [], final_label_arr = []):
#     # torch_cam_grad_cam_heatmap, torch_cam_grad_cam_class_output_probabilities = TorchCAM.compute_grad_cam(image, resnet_model, 'model.layer4.2.conv3', target_class_idx)
#     # torch_cam_grad_cam_heatmap = torch_cam_grad_cam_heatmap[0]
#     # print("before",torch_cam_grad_cam_heatmap.shape)
#     # torch_cam_grad_cam_heatmap = CommonUtils.get_resized_heatmap(torch_cam_grad_cam_heatmap, shape)
#     # print("after",torch_cam_grad_cam_heatmap.shape)
#     # if show_cams:
#     #     plt.matshow(torch_cam_grad_cam_heatmap)
#     #     plt.show()

#     # return torch_cam_grad_cam_heatmap

#     grad_cam_heatmap, _,  diff_grad_cam_heatmap, grad_cam_target_class_output_probabilities, grad_cam_ref_class_output_probabilities = GradCam.compute(image, resnet_model, last_conv_layer_name, target_class_idx, ref_class_idx)
#     grad_cam_heatmap = CommonUtils.get_resized_heatmap(grad_cam_heatmap, shape)
#     return grad_cam_heatmap
    

# for images, labels, bboxs in tqdm(test_loader):
#     images = images.to(device)
#     labels = labels.to(device)
    
#     # Iterate over each image in the batch
#     for i in range(images.size(0)):
#         # Select a single image and label
#         image = images[i:i+1]  # Keep batch dimension (1, C, H, W)
#         label = labels[i:i+1].cpu().item()
#         # print(image.shape)
#         bbox_coords = []
#         for idx, bbox in enumerate(bboxs):
#             # Convert tensor to a list or numpy array if needed
#             coords_arr = bbox.tolist() 
#             bbox_coords.append(coords_arr[i])

#         # filter the images with atleast one bounding box for calculating IOU.
#         if np.isnan(bbox_coords[0]) or np.isnan(bbox_coords[1]) or np.isnan(bbox_coords[2]) or np.isnan(bbox_coords[3]):
#             continue
        
#         bbox_coords = [int(x) for x in bbox_coords]

#         cams = get_cams(image, resnet_model, last_conv_layer_name, target_class_idx, ref_class_idx, show_cams=True, final_feature_arr=final_feature_arr, final_label_arr=final_label_arr)
#         print(cams.shape)
        
#         auprc = AUPRC.compute_auprc(bbox_coords, cams)
#         print(f"AUPRC: {auprc:.4f}")
        
#         break
#     break



# # for name, layer in model.named_modules():
# #     print(name)

#  # print(model.model.named_modules())
# # for name, layer in model.model.named_modules():
# #     print("layer here: ", name, layer)